## Purpose:
To train a factorization machine for recipe recommendation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### Training Regressor

In [ ]:
class FactorizationMachineRegression(nn.Module):
    def __init__(self, categorical_dims, num_continuous, embedding_dim):
        """
        Factorization Machine for Heterogeneous Data (Regression)

        Args:
            categorical_dims (list of int): List of vocabulary sizes for each categorical feature.
            num_continuous (int): Number of continuous features.
            embedding_dim (int): Dimensionality of the latent factors (k).
        """
        super(FactorizationMachineRegression, self).__init__()

        self.categorical_dims = categorical_dims
        self.num_continuous = num_continuous
        self.num_categorical = len(categorical_dims)
        self.total_features = self.num_categorical + num_continuous

        # Global bias (w0)
        self.bias = nn.Parameter(torch.zeros(1))

        # Linear weights (w_i)
        # Categorical linear weights via embedding layers (output size 1)
        self.cat_linear = nn.ModuleList([
            nn.Embedding(num_embeddings=dim, embedding_dim=1)
            for dim in categorical_dims
        ])
        # Continuous linear weights
        if num_continuous > 0:
            self.con_linear = nn.Linear(num_continuous, 1, bias=False)

        # Latent vectors / Factorization matrix (V)
        # Categorical latent vectors
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=dim, embedding_dim=embedding_dim)
            for dim in categorical_dims
        ])
        # Continuous latent vectors
        if num_continuous > 0:
            self.con_embeddings = nn.Parameter(torch.randn(num_continuous, embedding_dim))
            nn.init.xavier_uniform_(self.con_embeddings)

    def forward(self, x_cat, x_con=None):
        """
        Args:
            x_cat (Tensor): LongTensor of shape (batch_size, num_categorical)
            x_con (Tensor): FloatTensor of shape (batch_size, num_continuous)
        """
        batch_size = x_cat.size(0)

        # ----------------------------------------------------
        # 1. Linear Order (1st order interactions)
        # ----------------------------------------------------
        linear_part = torch.zeros(batch_size, 1, device=x_cat.device)

        # Process categorical linear parts
        for i in range(self.num_categorical):
            linear_part += self.cat_linear[i](x_cat[:, i])

        # Process continuous linear parts
        if self.num_continuous > 0 and x_con is not None:
            linear_part += self.con_linear(x_con)

        # ----------------------------------------------------
        # 2. Factorization Order (2nd order interactions)
        # ----------------------------------------------------
        # Gather all embeddings into a single tensor of shape (batch_size, total_features, embedding_dim)
        embeddings = []

        # Categorical embeddings: v_i * x_i (where x_i is implicitly 1)
        for i in range(self.num_categorical):
            embeddings.append(self.cat_embeddings[i](x_cat[:, i]).unsqueeze(1))

        # Continuous embeddings: v_j * x_j
        if self.num_continuous > 0 and x_con is not None:
            # x_con shape: (batch_size, num_continuous)
            # con_embeddings shape: (num_continuous, embedding_dim)
            # Resulting shape: (batch_size, num_continuous, embedding_dim)
            con_emb = x_con.unsqueeze(-1) * self.con_embeddings.unsqueeze(0)
            embeddings.append(con_emb)

        # Stack all interactions: (batch_size, total_features, embedding_dim)
        all_embeddings = torch.cat(embeddings, dim=1)

        # Mathematical trick for O(k*d) complexity:
        # 0.5 * sum( (sum(v_i*x_i))^2 - sum((v_i*x_i)^2) )
        sum_embeddings = torch.sum(all_embeddings, dim=1) # (batch_size, embedding_dim)
        sum_sq_embeddings = sum_embeddings ** 2

        sq_sum_embeddings = torch.sum(all_embeddings ** 2, dim=1) # (batch_size, embedding_dim)

        interaction_part = 0.5 * torch.sum(sum_sq_embeddings - sq_sum_embeddings, dim=1, keepdim=True)

        # ----------------------------------------------------
        # 3. Final Prediction
        # ----------------------------------------------------
        output = self.bias + linear_part + interaction_part
        return output.squeeze(-1) # Return shape (batch_size,)

In [ ]:
def prepare_data(recipes, reviews):
    # starting with skeleton of interaction matrix
    df = reviews[['DateSubmitted', 'AuthorId', 'RecipeId', 'Rating']]

    # sorting reviews by time for global time split
    df['DateSubmitted'] = pd.to_datetime(df['DateSubmitted'], utc=True).astype('int64') // 10**9
    df = df.rename(columns={'DateSubmitted':'Date_Unix'})
    df = df.sort_values(by='Date_Unix', ascending=True)

    # Adding pca feature vectors to interaction matrix
    # creating hash table connecting item id to item feature vector
    dict_recipes = {}
    for i in range(recipes.shape[0]):
        my_array = recipes.iloc[i].values
        dict_recipes[my_array[0]] = my_array[1:]
    # creating 2d array following format of interaction matrix
    interaction_recipes = np.empty((df.shape[0], 35))
    for i, r_id in enumerate(df['RecipeId']):
        interaction_recipes[i] = dict_recipes[r_id]

    # mapping AuthorId and RecipeId to new values so that they range from 0-N
    # keys are native ids, values are new ids ranging from 0-N
    a_ids = df['AuthorId'].unique()
    r_ids = df['RecipeId'].unique()

    a_id_mapper = dict()
    r_id_mapper = dict()

    for i, val in enumerate(a_ids):
        a_id_mapper[val] = i
    for i, val in enumerate(r_ids):
        r_id_mapper[val] = i

    df['AuthorId_mapped'] = df['AuthorId'].apply(lambda x: a_id_mapper[x])
    df['RecipeId_mapped'] = df['RecipeId'].apply(lambda x: r_id_mapper[x])

    X = np.hstack([df[['AuthorId_mapped', 'RecipeId_mapped']].values, interaction_recipes])
    y = df['Rating'].values

    return X, y, a_id_mapper, r_id_mapper

In [ ]:
# Loading recipes and reviews
data_path = '/content/drive/MyDrive/Colab Notebooks/Codefest-26 - Comcast_Dead_Zone/DSCI 641/data/'
recipes = pd.read_csv(data_path+'cleaned_data/top_recipes_pca.csv')
reviews = pd.read_csv(data_path+'cleaned_data/reviews_top_recipes.csv')

In [ ]:
# Preparing data
X, y, a_id_mapper, r_id_mapper = prepare_data(recipes, reviews)
reverse_a_id_mapper = {v: int(k) for k, v in a_id_mapper.items()}
reverse_r_id_mapper = {v: int(k) for k, v in r_id_mapper.items()}

# create train test splits
split_inds = [int(X.shape[0]*0.8), int(X.shape[0]*0.9)]
x_train, x_val, x_test = X[:split_inds[0]], X[split_inds[0]:split_inds[1]], X[split_inds[1]:]
y_train, y_val, y_test = y[:split_inds[0]], y[split_inds[0]:split_inds[1]], y[split_inds[1]:]

x_val_cat = torch.tensor(x_val[:,:2], dtype=torch.long)
x_val_num = torch.tensor(x_val[:,2:], dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

x_test_cat = torch.tensor(x_test[:,:2], dtype=torch.long)
x_test_num = torch.tensor(x_test[:,2:], dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

/tmp/ipykernel_1256/761686592.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['DateSubmitted'] = pd.to_datetime(df['DateSubmitted'], utc=True).astype('int64') // 10**9


In [ ]:
# Dataset Configuration
cat_features_dim = [189870, 23428]  # 2 categorical features with their unique value counts/num of categories
num_con_features = 35               # 35 continuous features
latent_dim = 32                     # Size of embedding factors

# use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate model
model = FactorizationMachineRegression(
    categorical_dims=cat_features_dim,
    num_continuous=num_con_features,
    embedding_dim=latent_dim
)

# move model to the GPU
model = model.to(device)

# Defining dataset and dataloader
x_train_cat = torch.tensor(x_train[:,:2], dtype=torch.long)
x_train_num = torch.tensor(x_train[:,2:], dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
dataset = TensorDataset(x_train_cat, x_train_num, y_train)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Training component setup
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: cuda


In [ ]:
# Training loop
epochs = 30
model.train()    # Set model to training mode
for epoch in range(epochs):
    for batch_cat, batch_num, batch_target in data_loader:
        optimizer.zero_grad()  # Clear previous gradients
        batch_cat = batch_cat.to(device, non_blocking=True)
        batch_num = batch_num.to(device, non_blocking=True)
        batch_target = batch_target.to(device, non_blocking=True)

        predictions = model(batch_cat, batch_num)      # compute predictions
        loss = criterion(predictions, batch_target)    # compute loss

        loss.backward()    # compute gradients
        optimizer.step()   # update model weights

    # Print progress every epoch
    val_preds = model(x_val_cat.to(device, non_blocking=True), x_val_num.to(device, non_blocking=True))
    val_loss = criterion(val_preds, y_val.to(device, non_blocking=True))
    print(f"Epoch [{epoch+1}/{epochs}] -- Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

Epoch [1/30] -- Train Loss: 1.1136, Val Loss: 16.5268
Epoch [2/30] -- Train Loss: 0.4735, Val Loss: 15.6479
Epoch [3/30] -- Train Loss: 1.1905, Val Loss: 15.8818
Epoch [4/30] -- Train Loss: 0.8314, Val Loss: 15.8039
Epoch [5/30] -- Train Loss: 0.4421, Val Loss: 15.7322
Epoch [6/30] -- Train Loss: 0.3180, Val Loss: 15.3961
Epoch [7/30] -- Train Loss: 0.3761, Val Loss: 15.2388
Epoch [8/30] -- Train Loss: 2.1283, Val Loss: 15.2381
Epoch [9/30] -- Train Loss: 0.2860, Val Loss: 17.9179
Epoch [10/30] -- Train Loss: 0.4203, Val Loss: 14.8816
Epoch [11/30] -- Train Loss: 0.2558, Val Loss: 14.7239
Epoch [12/30] -- Train Loss: 1.0459, Val Loss: 15.2479
Epoch [13/30] -- Train Loss: 0.0783, Val Loss: 15.6218
Epoch [14/30] -- Train Loss: 0.2043, Val Loss: 16.3293
Epoch [15/30] -- Train Loss: 2.7886, Val Loss: 17.7204
Epoch [16/30] -- Train Loss: 0.4570, Val Loss: 14.5953
Epoch [17/30] -- Train Loss: 0.1368, Val Loss: 14.8671
Epoch [18/30] -- Train Loss: 0.3841, Val Loss: 14.6816
Epoch [19/30] -- Tr

In [ ]:
# Saving weights
save_path = '/content/drive/MyDrive/Colab Notebooks/Codefest-26 - Comcast_Dead_Zone/DSCI 641/data/models/fmregressor_weights.pth'
torch.save(model.state_dict(), save_path)

In [ ]:
# Loading weights
model.load_state_dict(torch.load(save_path, weights_only=True))
model.eval()

FactorizationMachineRegression(
  (cat_linear): ModuleList(
    (0): Embedding(189870, 1)
    (1): Embedding(23428, 1)
  )
  (con_linear): Linear(in_features=35, out_features=1, bias=False)
  (cat_embeddings): ModuleList(
    (0): Embedding(189870, 32)
    (1): Embedding(23428, 32)
  )
)

In [ ]:
# Inspecting predictions on validation set
val_preds = model(x_val_cat.to(device, non_blocking=True), x_val_num.to(device, non_blocking=True))
val_loss = criterion(val_preds, y_val.to(device, non_blocking=True))

val_preds = val_preds.detach().cpu().numpy().reshape(-1, 1)
val_test = y_val.detach().cpu().numpy().reshape(-1, 1)

X_val = np.hstack((x_val_cat.detach().cpu().numpy(), val_preds, val_test))

In [ ]:
# Converting to pandas dataframe for ease
X_val = pd.DataFrame(X_val, columns=['Author_Id', 'Recipe_Id', 'Rating_Pred', 'Rating_True'])
X_val['Author_Id'] = X_val['Author_Id'].apply(lambda x: reverse_a_id_mapper[x])
X_val['Recipe_Id'] = X_val['Recipe_Id'].apply(lambda x: reverse_r_id_mapper[x])
X_val['Rating_Pred'] = X_val['Rating_Pred'].round().astype(int)
X_val['Rating_True'] = X_val['Rating_True'].astype(int)

X_val = X_val.sort_values(by=['Author_Id', 'Rating_True'], ascending=False)

In [ ]:
# Looking at users with at least reviews in the validataion set
my_counts = X_val.Author_Id.value_counts()
my_counts[my_counts.values == 10]

,count
Author_Id,
121690,10
2002714,10
1802501045,10
1339655,10
259154,10
...,...
149295,10
2901572,10
2667031,10


In [ ]:
# How did the model's predicted rating compare to the ground truth?
# Model did not do well
X_val[X_val.Author_Id == 149295]

,Author_Id,Recipe_Id,Rating_Pred,Rating_True
35444,149295,158671,1,5
35670,149295,77023,-2,5
36103,149295,110514,6,5
36746,149295,292430,2,5
43111,149295,63501,7,5
44440,149295,41950,2,5
50407,149295,66448,3,5
59880,149295,26380,0,5
35117,149295,103403,4,0
35308,149295,338863,6,0
